# Config 1: Gemma, CPU-only, zero-shot

The floor of the three-config comparison: an off-the-shelf small
open-weight model, no fine-tuning, no tool access, running entirely on
ordinary CPU hardware -- the air-gapped baseline. `config2-qlora-gpu/`
fine-tunes the same base model; `config3-frontier-skills/` gives a
frontier model SAS-metadata tool access. See the repo root's `PLAN.md`
for the full three-way framing.

This notebook is a runnable copy of `SETUP.md` in this folder (section
numbers match). It runs **either on Colab's free CPU runtime or on a box
with a local Ollama server**, picked automatically in section 3:

| | `--backend hf` (Colab) | `--backend ollama` (this box) |
|---|---|---|
| model source | `transformers` downloads `google/gemma-3-4b-it` | local Ollama server at `/internal/e2b-gemma` |
| weights | HF safetensors, bfloat16 | GGUF int4 (`gemma3:1b` / `gemma3:4b`) |
| base-model parity with config2 | exact (same repo id) | approximate (GGUF build of the same weights) |
| JSON-mode decoding | no | yes (`format: "json"`) |
| speed | slow -- measure before committing (section 4) | ~3-6 min/program measured for `gemma3:1b` |

**On Colab, leave Runtime -> Change runtime type on CPU (or "None").**
No accelerator is needed, and needing none is the entire point of config1
versus config2's T4 requirement.

Run top to bottom: get the repo -> install deps -> pick a backend ->
generate documentation -> score -> save the outputs off Colab ->
*(optional)* push results to SAS via ODA.

## 0. Get the whole repo (Colab)

Pull the **whole** repo, not just this folder: sections 4-5 need
`../eval-programs/` (the 20 held-out programs + gold) and `../results/`
(`score.py`, `run_eval.py`) as siblings of this folder. Pulling only
`config1-gemma-cpu/` leaves every `../eval-programs/...` and
`../results/...` path below failing with "No such file or directory".

Off Colab this cell is a no-op beyond confirming where it is running, so
the notebook is the same file in both places.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/patrickjlong1/sas-llm-paper.git"
CLONE_DIR = "/content/sas-llm-paper"

if IN_COLAB:
    if not os.path.isdir(CLONE_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, CLONE_DIR],
                       check=True)
    os.chdir(os.path.join(CLONE_DIR, "config1-gemma-cpu"))

REPO_ROOT = os.path.dirname(os.getcwd())
print("running on Colab:", IN_COLAB)
print("cwd:            ", os.getcwd())
for need in ("../eval-programs/programs", "../eval-programs/gold", "../results"):
    print("  %-28s %s" % (need, "OK" if os.path.isdir(need) else "MISSING"))

## 1. Python dependencies

`requirements.txt` covers both backends: `requests` for the Ollama HTTP
API, and `transformers`/`torch`/`accelerate`/`huggingface_hub` for the
`hf` backend.

The cell below probes for a local Ollama server first (stdlib only -- it
has to work before anything is installed) and installs **only what that
backend needs**. On a box that already serves the model, pulling ~2 GB of
torch to not use it is not a no-op. On Colab there's no server, so it
installs the full set; budget a couple of minutes.

The optional ODA push in section 7 additionally needs `saspy` + `pandas`;
that section installs them itself.

In [ ]:
import urllib.error
import urllib.request

def _ollama_up(url="http://127.0.0.1:11434/api/version", timeout=5):
    """stdlib only: this runs before `requests` is guaranteed installed."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status == 200
    except (urllib.error.URLError, OSError, ValueError):
        return False

HAS_OLLAMA = _ollama_up()
print("local Ollama server reachable:", HAS_OLLAMA)

In [ ]:
if HAS_OLLAMA:
    # Ollama does the model work; document_sas.py only needs an HTTP client.
    !pip -q install requests
else:
    # No server -- the hf backend loads the weights in-process.
    !pip -q install -r requirements.txt

## 2. The model runtime

**`--backend ollama` (this box).** The weights + Ollama server live
OUTSIDE this folder, at `/internal/e2b-gemma/` -- infrastructure (a 9.6 GB
directory of model blobs and platform binaries), not project code, so it
isn't duplicated here. See `/internal/e2b-gemma/README.md`.

`gemma3:1b` is this backend's default. `gemma3:4b` was tried first (to
match config2's 4B QLoRA base) but is a MULTIMODAL checkpoint -- the
Ollama server log shows it loading an `mmproj` blob and running an
image-size warmup pass -- and it timed out loading on this box's 4.7 GB
RAM: 13m47s elapsed, cycling `loading model` / `not responding` under
swap pressure, until Ollama's own load-timeout killed it and
`document_sas.py` surfaced a `500 Internal Server Error`. `gemma3:1b` is
text-only, lighter, and is what the ~11 tok/s prompt / ~4.5 tok/s
generation / 3-6 min-per-program numbers in this project were measured
against. With more RAM, pass `--model gemma3:4b` yourself to restore
exact base-model-size parity with config2 (and say so in the paper -- that
parity is the point of the config1-vs-config2 comparison), and raise
`OLLAMA_LOAD_TIMEOUT` before `ollama serve` if it just needs more time.

**`--backend hf` (Colab).** No Ollama server exists there. `transformers`
downloads `google/gemma-3-4b-it` -- the exact weights config2 fine-tunes
-- and runs them on CPU in bfloat16. Nothing to start; the next cell will
simply report no server, which is expected and fine.

In [ ]:
# Detail on the server section 1 already probed. Expected to print nothing
# useful on Colab; on a box with Ollama it prints a version string.
!curl -s -m 5 http://127.0.0.1:11434/api/version || echo "no local Ollama server (expected on Colab)"

In [ ]:
# Only needed on this box if the check above returned nothing (e.g. after a
# reboot). Starts the server in the background and detaches. Safe to leave
# commented out; it does nothing on Colab.
# !nohup /internal/e2b-gemma/serve.sh > /internal/e2b-gemma/server.log 2>&1 &
# !disown

## 3. Pick the backend

Auto: `hf` on Colab, `ollama` anywhere a local server answered. Override
`BACKEND` / `MODEL` / `DTYPE` in the cell below if you want the other one.

`google/gemma-3-4b-it` is **gated**: accept the license on its Hugging
Face page while logged in, then run the auth cell. Prefer Colab's Secrets
panel (padlock icon, left sidebar) with an `HF_TOKEN` secret --
`login()` picks that up and only prompts if it isn't set. This is the
same gate config2 goes through for the same weights.

In [ ]:
import json
import shutil

# HAS_OLLAMA was probed in section 1, before anything was installed.
# Force the other backend by setting it yourself here, e.g. BACKEND = "hf".
BACKEND = "ollama" if HAS_OLLAMA else "hf"

# --model/--dtype defaults, chosen for the backend. Override either to
# trade fidelity for speed -- see the note printed below.
MODEL = {"ollama": "gemma3:1b", "hf": "google/gemma-3-4b-it"}[BACKEND]
DTYPE = "auto"          # hf only: auto == bfloat16. See document_sas.py --dtype.

# Output/catalog dirs are per-backend on purpose: the two paths are
# different models on different serving stacks, so pooling them into one
# results-table row would mix two configs.
SUFFIX = "" if BACKEND == "ollama" else "-hf"
RUN = "config1-gemma-cpu" + SUFFIX
OUT = "../results/preds/" + RUN
CAT = "../results/catalog/" + RUN
PREFIX = "../results/outputs/" + RUN        # run_eval.py --out-prefix
SCORES = PREFIX + ".scores.jsonl"

# Set as env vars as well as Python names so the `!` cells below read the
# same way if you paste them into a terminal. (Keep every shell reference a
# bare `$NAME` followed by whitespace: IPython's `$` expansion swallows a
# following dot as attribute access, so `$PREFIX.scores.jsonl` would break --
# hence the separate SCORES variable.)
os.environ.update(BACKEND=BACKEND, MODEL=MODEL, DTYPE=DTYPE, OUT=OUT, CAT=CAT,
                  RUN=RUN, PREFIX=PREFIX, SCORES=SCORES)
print("backend=%s  model=%s  dtype=%s" % (BACKEND, MODEL, DTYPE))
print("predictions -> %s" % OUT)
if BACKEND == "hf":
    print("\nNOTE: 4B in bfloat16 on a free Colab CPU is memory-feasible but slow "
          "(PyTorch has no fast bf16 CPU kernels without AVX512-BF16). If the "
          "timing in section 4 is impractical, the documented faster option is "
          "MODEL='google/gemma-3-1b-it' with DTYPE='float32' -- report it as such, "
          "since it breaks exact base-model parity with config2.")

In [ ]:
# Only needed for BACKEND == "hf". No-op if HF_TOKEN is already set.
if BACKEND == "hf":
    from huggingface_hub import login
    login()

## 4. Run it

Zero-shot: no fine-tuning happens here (that's config2). On the `ollama`
backend, `format: "json"` constrains decoding to valid JSON *syntax*,
which helps a lot on a 1B model; `transformers` has no equivalent, so the
`hf` backend produces more schema-invalid output on the same prompt.
Either way `schema.py`'s `validate()` still checks the CONTENT shape, and
`results/score.py` reports schema validity as its own metric.

Each program writes `<name>.pred.json` (the structured dictionary),
`<name>.meta.json` (timing/cost, in the shape `results/run_eval.py`
expects), `<name>.raw.txt` (the raw response, kept even on a parse
failure), and `<name>.guardrail.json` (hallucination check against
`extract.py`'s static source scan). Every run also upserts into a local,
offline JSONL catalog (`catalog.py`) that section 7 can push to SAS.

**One program first** -- both to see the output shape and to get a real
per-program time before committing to all 20. The cell after it does the
arithmetic for you.

In [ ]:
!python3 document_sas.py ../eval-programs/programs/prog900_estab.sas \
    --backend $BACKEND --model $MODEL --dtype $DTYPE --out $OUT --catalog $CAT

In [ ]:
# How long would all 20 take at the speed just measured? Colab free
# sessions idle-disconnect after ~90 minutes and cap around 12 hours, so
# decide here rather than discovering it eight hours in.
meta_path = os.path.join(OUT, "prog900_estab.meta.json")
if os.path.exists(meta_path):
    secs = json.load(open(meta_path))["elapsed_sec"]
    print("measured: %.1f s for 1 program" % secs)
    print("estimated: %.1f min for all 20 (model load already paid once)"
          % (secs * 20 / 60.0))
    if secs * 20 > 3 * 3600:
        print("\n>3h for the full set. Options: run the batch cell with --limit N "
              "and report a partial run AS partial, switch to the faster "
              "1B/float32 combination in section 3, or run the full set "
              "somewhere without a session cap.")
else:
    print("run the cell above first")

Then the whole eval set -- one program at a time, in sequence (no GPU, no
request batching on either backend). A per-file failure (bad file, Ollama
hiccup, timeout, OOM) is logged and skipped rather than aborting the run.

Add `--limit N` for a partial smoke run. A partial run is **not** a
result: `score.py` scores against all 20 gold files and counts every
missing prediction as a program that produced no output, so a 5-program
run scores like a 20-program run that failed 15 times.

In [ ]:
!python3 document_sas.py --dir ../eval-programs/programs \
    --backend $BACKEND --model $MODEL --dtype $DTYPE --out $OUT --catalog $CAT

## 5. Score this run

`results/run_eval.py` wraps `score.py` into the `.scores.jsonl` the plan's
results table reads, and stamps a `.provenance.json` sidecar recording
exactly which eval corpus was scored -- `--table` marks a row **STALE**
rather than printing numbers that no longer refer to the corpus on disk.
(That check exists because it already went wrong once; see
`../results/outputs/stale-2026-09-17/README.md`.)

`--run-judge` is deliberately left off: the default judge is a local
Ollama model, which Colab doesn't have. Run the judge later on a box that
does; until then the table prints `not run` for Description score rather
than a made-up number.

In [ ]:
!python3 ../results/run_eval.py --config $RUN --pred-dir $OUT --out-prefix $PREFIX

In [ ]:
!python3 ../results/run_eval.py --table \
    --scores $SCORES --meta-dir $OUT \
    --label "Config 1: Gemma CPU ($BACKEND, $MODEL)"

## 6. Get the outputs off Colab

Colab's local disk does **not** survive a runtime recycle, and neither do
the downloaded weights. Unlike config2 there's no trained artifact to
lose here -- but the predictions, catalog and scores ARE the run, and
re-creating them costs the whole generation time again.

Skip this cell off Colab.

In [ ]:
if IN_COLAB:
    from google.colab import files
    import shutil
    for src, name in ((OUT, RUN + "-preds"),
                      (CAT, RUN + "-catalog"),
                      ("../results/outputs", RUN + "-scores")):
        if os.path.isdir(src):
            shutil.make_archive("/content/" + name, "zip", src)
            files.download("/content/%s.zip" % name)
else:
    print("not on Colab -- outputs are already on local disk")

## 7. (Optional) Push the catalog into SAS via ODA

Only needed to materialize the generated dictionary as real SAS datasets
(`push_to_oda.py`). Config1 has no ground-truth SAS access **by design**
-- that asymmetry is config3's job and part of the finding -- so nothing
here feeds back into config1's own output. Skip this section entirely if
you only want the local JSON/catalog output.

**Get a free ODA account** first if you don't have one (SAS OnDemand for
Academics signup -- no cost, academic/non-commercial use).

**Credentials.** On your own machine, create `~/.authinfo` yourself, in a
terminal, never in a saved notebook cell:

```bash
echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
chmod 600 ~/.authinfo
```

On Colab, use the Secrets panel (padlock icon) with `ODA_USER` /
`ODA_PASS` secrets -- the cell below reads them from there and writes
`~/.authinfo` into the ephemeral session, prompting only if the secrets
aren't set. Either way the password never becomes part of the saved
notebook.

**Java.** `config/sascfg_personal.py` prefers the portable JRE at the repo
root (`../jre/`) and falls back to whatever `java` is on `PATH`. `jre/` is
136 MB and gitignored, so a fresh clone (i.e. Colab) does **not** have it
-- install a JDK there, which the cell below does.

**Region.** `config/sascfg_personal.py`'s `iomhost` list is already filled
in for a US-region/usw2 account; see `SETUP.md` for the Europe and Asia
Pacific host names.

In [ ]:
if IN_COLAB:
    !apt-get -qq install default-jdk > /dev/null
    !pip -q install saspy pandas
print("java:", shutil.which("java") or "NOT ON PATH (repo jre/ will be used if present)")

In [ ]:
import getpass

authinfo = os.path.expanduser("~/.authinfo")
if os.path.exists(authinfo):
    print("~/.authinfo found -- leaving it alone")
else:
    user = password = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            user, password = userdata.get("ODA_USER"), userdata.get("ODA_PASS")
        except Exception:
            pass
    user = user or input("ODA email: ")
    password = password or getpass.getpass("ODA password: ")
    with open(authinfo, "w") as fh:
        fh.write("oda user %s password %s\n" % (user, password))
    os.chmod(authinfo, 0o600)
    del password
    print("wrote", authinfo, "(chmod 600)")

In [ ]:
# Writes PROGRAM_SUMMARY, MACRO_PARAMS, DATA_DICTIONARY into your SASUSER
# library (SAS's auto-assigned, persistent-across-sessions library -- no
# LIBNAME statement or path to know). Pass --libname/--libpath only for a
# different, custom-path library.
#
# saspy lives in a separate venv on this box but in the notebook kernel on
# Colab, so pick whichever interpreter actually has it.
VENV_PY = "/internal/venvs/main/bin/python3"
PY = VENV_PY if (not IN_COLAB and os.path.exists(VENV_PY)) else sys.executable
os.environ["PY"] = PY
print("using interpreter:", PY)

In [ ]:
!$PY push_to_oda.py --catalog $CAT

## Why zero-shot, not few-shot or fine-tuned

Tested in-context learning (showing the model 1-2 example SAS->JSON
pairs) here and it made output WORSE, not better: with even one exemplar
in context, `gemma3:1b` lost coherence and fabricated an entirely
fictional SAS program instead of documenting the real one. That's a
genuine capability ceiling of a 1B-parameter model on this task, not a
prompt bug -- if you want few-shot or fine-tuned quality, that's config2
(`../config2-qlora-gpu/`).

## Known limitations (report these honestly in the results)

- Small-model JSON-mode output frequently fails `schema.py`'s structural
  validation outright (missing keys, wrong types, a list entry that's a
  bare string instead of an object). `document_sas.py` does NOT crash on
  this -- it records `schema_valid: false` and moves on, exactly so
  `results/score.py` can report schema validity as its own honest metric
  rather than one bad program aborting a batch run. The `hf` backend has
  no JSON-mode decoding constraint at all, so expect this to be worse
  there than on `ollama`.
- Even when schema-valid, meanings/types are genuinely best-guess and
  were observed wrong on toy synthetic input (e.g. calling a sampling
  weight "Work Time"). The guardrail only catches INVENTED names, never
  wrong MEANINGS -- read every entry, don't just trust an absence of
  flags.
- `extract.py`'s regex scan is a heuristic on arbitrary real SAS syntax
  -- it can miss real identifiers written in forms it doesn't anticipate,
  which under-flags (reports clean when it isn't).
- The two backends are not interchangeable for reporting. Same prompt and
  schema, but different weights format (GGUF int4 vs bfloat16), different
  parameter counts by default (1B vs 4B), and JSON-mode on one and not
  the other. Score them as two rows, not one -- which is why section 3
  writes them to different directories.
- A `--limit`ed run is a smoke test, not a result. `score.py` counts every
  un-predicted program as a failure, so a partial run's numbers are not
  comparable to a full one's.